In [222]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [223]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# DATASET

In [224]:
# load the cleaned dataset from notebook1
data = pd.read_csv("/kaggle/input/datasets/filippotenani/uniting-cleaned/uniting_cleaned.csv") 

In [225]:
data.columns

Index(['Creator name', 'Creator_gender', 'Filename', 'Social permalink',
       'Channel', 'Followers', 'Type of content', 'Post creation date',
       'Post creation time', 'Post caption', 'Reach', 'Likes', 'Comments',
       'Total clicks', 'Brand name', 'Industry', 'Local', 'Brand_SM',
       'media_duration_sec', 'face_frame_ratio', 'first_face_position_ratio',
       'motion_level', 'saturation', 'luminance', 'contrast', 'sharpness',
       'color_complexity'],
      dtype='object')

# FEATURE ENGINEERING

Now we create new features based on the existing ones

## TIME SLOT

We consider time slots trying to group similar hours together:
- night: 1-2-3-4-5
- morning: 6-7-8-9-10
- lunch: 11-12-13-14
- afternoon: 15-16-17
- dinner/aperitivo: 18-19-20
- evening: 21-22-23-24

Then how do we handle the NaN values (00:00:00, 01:00:00, ...)???
The professor said to keep them exactly as they are so we don't consider them NaN

In [228]:
data["Post creation time"].dtypes

dtype('O')

In [229]:
# Extract the hour as integer from the first two characters of the string
data['hour'] = data['Post creation time'].str[:2].astype(int)

# Map the hour to the corresponding time slot
def map_fascia(hour):
    if hour in range(0, 6):
        return 'notte'
    elif hour in range(6, 11):
        return 'mattina'
    elif hour in range(11, 15):
        return 'pranzo'
    elif hour in range(15, 18):
        return 'pomeriggio'
    elif hour in range(18, 21):
        return 'cena'
    else:  # 21-23
        return 'sera'

data['fascia_oraria'] = data['hour'].map(map_fascia)

# Reposition the column right after 'Post creation time'
col_idx = data.columns.get_loc('Post creation time') + 1
data.insert(col_idx, 'fascia_oraria', data.pop('fascia_oraria'))

# Remove the intermediate column
data.drop(columns=['hour'], inplace=True)

data['fascia_oraria'].value_counts()

fascia_oraria
pranzo        1293
pomeriggio     858
cena           516
notte          256
mattina        183
sera           137
Name: count, dtype: int64

In [230]:
print(data[['Post creation time', 'fascia_oraria']].head(100).to_string())

   Post creation time fascia_oraria
0            17:23:02    pomeriggio
1            20:32:40          cena
2            18:14:01          cena
3            12:13:38        pranzo
4            21:45:28          sera
5            01:00:00         notte
6            21:27:07          sera
7            21:57:05          sera
8            20:10:28          cena
9            12:46:34        pranzo
10           12:05:19        pranzo
11           01:00:00         notte
12           16:27:30    pomeriggio
13           08:24:31       mattina
14           18:35:34          cena
15           23:25:59          sera
16           12:29:08        pranzo
17           15:06:13    pomeriggio
18           10:11:00       mattina
19           02:00:00         notte
20           00:00:00         notte
21           01:00:00         notte
22           12:39:59        pranzo
23           20:39:46          cena
24           18:40:29          cena
25           11:06:34        pranzo
26           13:33:59       

## MONTH

We extract the month from 'Post creation date' which will be a category. We start with one category per month (January, February,...), then if we see in the exploratory analysis that it makes sense to group them we will group them.

In [232]:
data["Post creation date"].head()

0    2024-05-15
1    2025-09-20
2    2025-07-02
3    2024-06-13
4    2025-10-30
Name: Post creation date, dtype: object

In [233]:
# Extract the month as integer from characters 5-7 of the date string
data['month'] = data['Post creation date'].str[5:7].astype(int)

# Map the month to the corresponding name
def map_mese(month):
    if month == 1:
        return 'gennaio'
    elif month == 2:
        return 'febbraio'
    elif month == 3:
        return 'marzo'
    elif month == 4:
        return 'aprile'
    elif month == 5:
        return 'maggio'
    elif month == 6:
        return 'giugno'
    elif month == 7:
        return 'luglio'
    elif month == 8:
        return 'agosto'
    elif month == 9:
        return 'settembre'
    elif month == 10:
        return 'ottobre'
    elif month == 11:
        return 'novembre'
    else:  # 12
        return 'dicembre'

data['mese'] = data['month'].map(map_mese)

# Reposition the column right after 'Post creation date'
col_idx = data.columns.get_loc('Post creation date') + 1
data.insert(col_idx, 'mese', data.pop('mese'))

# Remove the intermediate column
data.drop(columns=['month'], inplace=True)

data['mese'].value_counts()

mese
settembre    459
giugno       451
ottobre      425
maggio       361
luglio       332
novembre     269
febbraio     264
aprile       202
dicembre     174
marzo        171
gennaio       71
agosto        64
Name: count, dtype: int64

In [234]:
print(data[['Post creation date', 'mese']].head(100).to_string())

   Post creation date       mese
0          2024-05-15     maggio
1          2025-09-20  settembre
2          2025-07-02     luglio
3          2024-06-13     giugno
4          2025-10-30    ottobre
5          2024-11-02   novembre
6          2025-05-17     maggio
7          2025-02-20   febbraio
8          2025-09-19  settembre
9          2024-11-03   novembre
10         2024-10-11    ottobre
11         2024-11-02   novembre
12         2025-02-24   febbraio
13         2024-09-23  settembre
14         2025-10-19    ottobre
15         2025-06-25     giugno
16         2024-09-23  settembre
17         2024-05-15     maggio
18         2025-05-19     maggio
19         2024-09-04  settembre
20         2024-05-18     maggio
21         2024-11-02   novembre
22         2025-07-10     luglio
23         2024-06-07     giugno
24         2025-08-25     agosto
25         2025-07-14     luglio
26         2025-06-25     giugno
27         2025-02-02   febbraio
28         2025-05-15     maggio
29        

## WEEKEND/SETTIMANALE

We separate weekdays and weekends, weekend includes Friday because we expect many posts on Friday evening

In [236]:
# Convert the date column to datetime format
data['Post creation date'] = pd.to_datetime(data['Post creation date'])

# Create the new column 'Giorno_della_settimana' based on the day
data['Weekend/Settimanale'] = np.where(
    # dt.dayofweek returns a number from 0 (Monday) to 6 (Sunday)
    # so if the returned number is less than or equal to 3 (Thursday) we consider it a weekday
    data['Post creation date'].dt.dayofweek <= 3, 
    'settimanale', 
    'weekend'
)

# Reposition the column right after 'mese'
col_idx = data.columns.get_loc('mese') + 1
data.insert(col_idx, 'Weekend/Settimanale', data.pop('Weekend/Settimanale'))

# Display the first results to verify
print(data[['Post creation date', 'Weekend/Settimanale']].head(100))

   Post creation date Weekend/Settimanale
0          2024-05-15         settimanale
1          2025-09-20             weekend
2          2025-07-02         settimanale
3          2024-06-13         settimanale
4          2025-10-30         settimanale
5          2024-11-02             weekend
6          2025-05-17             weekend
7          2025-02-20         settimanale
8          2025-09-19             weekend
9          2024-11-03             weekend
10         2024-10-11             weekend
11         2024-11-02             weekend
12         2025-02-24         settimanale
13         2024-09-23         settimanale
14         2025-10-19             weekend
15         2025-06-25         settimanale
16         2024-09-23         settimanale
17         2024-05-15         settimanale
18         2025-05-19         settimanale
19         2024-09-04         settimanale
20         2024-05-18             weekend
21         2024-11-02             weekend
22         2025-07-10         sett

## FACE

Create a variable that says whether a face appears in the video. This variable is 0 if face_frame_ratio=0, 1 otherwise.
This is because we believe there is a strong difference between videos without a face and videos where a face appears even briefly

In [238]:
# Create the new column 'faccia' based on whether 'face_frame_ratio' is 0 or not
# 'face_frame_ratio' indicates the percentage of video where the face is present, if 0 it means there's never a face
# the new variable 'faccia' is categorical {0,1} where 0 means no face and 1 means face
data['faccia'] = np.where(data['face_frame_ratio'] == 0, 0, 1)

# Reposition the column right after 'face_frame_ratio'
col_idx = data.columns.get_loc('face_frame_ratio') + 1
data.insert(col_idx, 'faccia', data.pop('faccia'))

# Display the first results to verify
print(data[['face_frame_ratio', 'faccia']].head(100))

    face_frame_ratio  faccia
0           1.000000       1
1           0.000000       0
2           0.000000       0
3           0.000000       0
4           1.000000       1
5           0.000000       0
6           0.500000       1
7           0.000000       0
8           0.000000       0
9           1.000000       1
10          1.000000       1
11          0.666667       1
12          1.000000       1
13          0.000000       0
14          0.333333       1
15          1.000000       1
16          0.333333       1
17          1.000000       1
18          1.000000       1
19          1.000000       1
20          0.000000       0
21          0.600000       1
22          0.000000       0
23          0.000000       0
24          1.000000       1
25          0.000000       0
26          1.000000       1
27          1.000000       1
28          0.000000       0
29          0.666667       1
30          0.666667       1
31          1.000000       1
32          0.666667       1
33          0.

## COGNITIVE_OVERLOAD

We create a feature called 'overload' that considers the visual overload the user experiences from the video.
The overload combines color_complexity and motion_level, i.e. if the video has a lot of motion and chaotic colors, it increases the cognitive load on the viewer.
We hypothesize that this could reduce engagement, in any case it can be useful to consider a single variable that combines the two, then through models we will verify the hypothesis.

These considerations on overload are partly reported in the following paper.
The paper mainly discusses visual complexity as impacting cognitive overload, for this reason we included saturation and motion_level.

https://ojs.ikm.mk/index.php/kij/article/view/7757

Also the following paper shows that highly saturated colors reduce attention:
"However, intensely bright and highly saturated colors such as red or purple may cause visual distractions that interfere with attention and information processing".

https://onlinelibrary.wiley.com/doi/10.1002/col.70040?af=R#:~:text=Research%20has%20shown%20that%20specific,interfaces%20%5B28%2D30%5D

Also as later confirmed in the exploratory analysis, we notice that saturation and motion_level are the only two "video quality" indicators negatively correlated with the target (engagement rate), while the other 4 have a positive correlation (except sharpness which has a positive correlation close to 0). So we think it makes sense to combine those two variables into a single one

In [241]:
# we compute cognitive overload by combining saturation and motion_level
data["cognitive_overload"] = data["saturation"] + data["motion_level"]

data[["saturation", "motion_level", "cognitive_overload"]].head()

,saturation,motion_level,cognitive_overload
0,0.545074,0.949434,1.494508
1,0.321415,0.924965,1.246380
2,0.744234,0.000394,0.744628
3,0.210730,0.282641,0.493371
4,0.334941,0.819924,1.154865


## FLASHINESS

Similarly to what we did with overload, we select color_complexity, contrast, luminance and put them together in a variable called flashiness since they are all positively correlated with engagement.

Sharpness instead has a positive correlation with engagement but close to 0, so practically uncorrelated with engagement, so we keep it separate on its own.

In [242]:
# the three variables have very different scales from each other
data[["color_complexity", "contrast", "luminance"]].describe()

# so we scale contrast and luminance between 0 and 1 which is approximately the range of color_complexity
data["contrast_scaled"] = (data["contrast"] - data["contrast"].min()) / (data["contrast"].max() - data["contrast"].min())
data["luminance_scaled"] = (data["luminance"] - data["luminance"].min()) / (data["luminance"].max() - data["luminance"].min())

# let's visualize the distribution after scaling
data[["color_complexity", "contrast_scaled", "luminance_scaled"]].describe()

,color_complexity,contrast_scaled,luminance_scaled
count,3243.000000,3243.000000,3243.000000
mean,0.644899,0.563827,0.495766
std,0.105378,0.103838,0.111614
min,0.000000,0.000000,0.000000
25%,0.581349,0.500108,0.443343
50%,0.657266,0.564263,0.507248
75%,0.719360,0.628291,0.561073
max,0.912459,1.000000,1.000000


In [243]:
# we compute flashiness by summing color_complexity, contrast and luminance
data["flashiness"] = data["color_complexity"] + data["contrast_scaled"] + data["luminance_scaled"]

data[["color_complexity", "contrast_scaled", "luminance_scaled", "flashiness"]].head()

,color_complexity,contrast_scaled,luminance_scaled,flashiness
0,0.448271,0.451793,0.484980,1.385044
1,0.797086,0.472465,0.346694,1.616246
2,0.552016,0.545923,0.088239,1.186177
3,0.708601,0.691404,0.541291,1.941295
4,0.640383,0.398996,0.311870,1.351249


In [244]:
# we remove the scaled variables because they are no longer needed
data.drop(columns=["contrast_scaled", "luminance_scaled"], inplace=True)

QUESTION FOR THE PROFESSOR: is it ok to scale luminance and contrast? or do we just multiply color_complexity by 100???

We will also later verify with models whether what we found in the following papers is true: high color complexity increases engagement:

https://news.nd.edu/news/high-color-complexity-in-social-media-images-proves-more-eye-catching-increases-user-engagement/

https://phys.org/news/2024-10-complexity-social-media-engagement.html

# TARGETS

To compute the targets we separate the dataset into INSTAGRAM_STORY on one side and INSTAGRAM_REEL + TIKTOK_POST on the other, because due to data availability we compute the target differently for the two datasets

In [247]:
story_data = data[data["Type of content"] == "INSTAGRAM_STORY"]
reel_data = data[data["Type of content"].isin(["INSTAGRAM_REEL", "TIKTOK_POST"])]

In [248]:
story_data["Type of content"].value_counts()
reel_data["Type of content"].value_counts()

Type of content
INSTAGRAM_REEL    685
TIKTOK_POST       328
Name: count, dtype: int64

## PERCENTAGE OF FOLLOWERS REACHED

percentage of followers reached = reach/followers

We compute it for both content types, it shows the percentage of followers reached

In [250]:
# we compute percentage of followers reached for both stories and reels, this target variable is shared by both
story_data["PERC_REACHED"] = story_data["Reach"] / story_data["Followers"]
reel_data["PERC_REACHED"] = reel_data["Reach"] / reel_data["Followers"]

/tmp/ipykernel_55/1644301220.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  story_data["PERC_REACHED"] = story_data["Reach"] / story_data["Followers"]
/tmp/ipykernel_55/1644301220.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reel_data["PERC_REACHED"] = reel_data["Reach"] / reel_data["Followers"]


In [251]:
story_data["PERC_REACHED"].head(100)
reel_data["PERC_REACHED"].head(100)

# we notice that on average reels seem to have much higher percentage_reached compared to stories since the latter only last 24h

40       1.220096
55      29.692926
60       0.060450
81      23.219871
113      0.065250
120     13.170161
126      0.424059
151      3.144759
168     26.993569
181      0.022568
184      0.428158
187     24.712605
193      0.422821
201      0.098323
256      4.845338
316     15.918103
332     48.588424
345     13.162058
373     18.061383
441     43.189711
453      0.268728
464     13.019003
537      0.542572
569      0.426701
597      0.159656
814      0.051640
844      0.175532
900      4.286592
912      0.343979
930      0.210143
995     13.062186
1024     0.605680
1025     0.070248
1038     1.301225
1076     0.264461
1101     0.185281
1106     0.360043
1116    18.647170
1126     0.610088
1127     0.063079
1153     1.138096
1178     0.878818
1191     1.523397
1217     0.101640
1225     0.658297
1243    17.135531
1245     0.094514
1364     0.217519
1371     0.446596
1388     1.777828
1390     0.113323
1402     0.328482
1421     0.027903
1427     0.419228
1429     0.154884
1439     0

## ENGAGEMENT RATE

engagement rate = (alpha*Likes + beta*Comments) / Reach

Computed only for reels/tiktok, it shows how many likes and comments were generated as a percentage of reach.
According to the following paper, users make "more effort" to leave a comment compared to a like, so we will give comments a weight 'beta' greater than likes

https://www.researchgate.net/publication/228237594_Can_You_Measure_the_ROI_of_Your_Social_Media_Marketing

We therefore want to determine the weight of comments relative to likes. We found a leaked document about how the Facebook algorithm works, it's not the Instagram algorithm which is what we need for our dataset, but we think it's a good proxy, especially since this weight value was very difficult to find/quantify. The following leaked document shows how likes count 1 point, while comments count from 15 to 30 points depending on their length, where single-character comments don't count at all.

https://www.deeplearning.ai/the-batch/how-facebook-fills-the-feed/

So we consider alpha=1 and beta=20 (20 as a sort of middle ground between 15-30, also considering that single-character comments are worth zero)

In [253]:
alpha = 1
beta = 20
reel_data["ENGAGE_RATE"] = (alpha*reel_data["Likes"] + beta*reel_data["Comments"]) / reel_data["Reach"]

reel_data["ENGAGE_RATE"].head(100)

/tmp/ipykernel_55/3445411956.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reel_data["ENGAGE_RATE"] = (alpha*reel_data["Likes"] + beta*reel_data["Comments"]) / reel_data["Reach"]


40      0.047569
55      0.000410
60      0.047872
81      0.005633
113     0.255423
120     0.001787
126     0.013071
151     0.002045
168     0.004224
181     0.029402
184     0.012378
187     0.001364
193     0.003425
201     0.038316
256     0.013617
316     0.005246
332     0.004319
345     0.001253
373     0.001365
441     0.001045
453     0.062229
464     0.003552
537     0.013512
569     0.072919
597     0.014518
814     0.081569
844     0.035710
900     0.000945
912     0.022226
930     0.109925
995     0.001356
1024    0.039279
1025    0.037350
1038    0.001326
1076    0.041396
1101    0.029642
1106    0.029742
1116    0.005161
1126    0.029389
1127    0.050124
1153    0.021506
1178    0.022072
1191    0.002470
1217    0.053780
1225    0.048667
1243    0.004511
1245    0.039289
1364    0.050519
1371    0.052427
1388    0.064616
1390    0.096701
1402    0.069516
1421    0.222055
1427    0.055867
1429    0.081161
1439    0.061568
1466    0.016911
1471    0.100524
1473    0.0553

## CLICK-THROUGH RATE

click-through rate = Total clicks / Reach

Computed only for stories, it shows the percentage of reached users who click on the sponsor link

In [254]:
story_data["CTR"] = story_data["Total clicks"] / story_data["Reach"]

story_data["CTR"].head(100)

/tmp/ipykernel_55/314109763.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  story_data["CTR"] = story_data["Total clicks"] / story_data["Reach"]


0      0.023508
1      0.001667
2      0.009435
3      0.008250
4      0.001438
5      0.004310
6      0.000282
7      0.000473
8      0.000894
9      0.001875
10     0.033352
11     0.006404
12     0.000561
13     0.008333
14     0.005252
15     0.001546
16     0.008250
17     0.002013
18     0.008250
19     0.005450
20     0.008082
21     0.000753
22     0.002561
23     0.002175
24     0.008353
25     0.005014
26     0.002183
27     0.000450
28     0.000770
29     0.000588
30     0.005431
31     0.008200
32     0.005244
33     0.001521
34     0.010333
35     0.008267
36     0.003544
37     0.000486
38     0.003579
39     0.003770
41     0.008333
42     0.000388
43     0.024377
44     0.009615
45     0.000925
46     0.001618
47     0.002965
48     0.000915
49     0.001200
50     0.025388
51     0.000670
52     0.008200
53     0.016020
54     0.003195
56     0.007313
57     0.008400
58     0.000813
59     0.001560
61     0.006825
62     0.000497
63     0.008000
64     0.008250
65     0

## COMMENT PER LIKE

comment_per_like = Comments / Likes

Computed only for reels/tiktok, it measures how much the content stimulates discussion compared to simple passive approval

In [255]:
reel_data["COMM_PER_LIKE"] = reel_data["Comments"] / reel_data["Likes"]

reel_data["COMM_PER_LIKE"].head(100)

/tmp/ipykernel_55/2886904767.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reel_data["COMM_PER_LIKE"] = reel_data["Comments"] / reel_data["Likes"]


40      0.015162
55      0.013378
60      0.040000
81      0.048356
113     0.010949
120     0.007911
126     0.011295
151     0.012500
168     0.013186
181     0.009494
184     0.005469
187     0.020053
193     0.000000
201     0.000000
256     0.011364
316     0.013746
332     0.013902
345     0.012107
373     0.015332
441     0.011374
453     0.027350
464     0.023517
537     0.010638
569     0.004949
597     0.027211
814     0.000000
844     0.030268
900     0.009434
912     0.012807
930     0.011295
995     0.017032
1024    0.140659
1025    0.000000
1038    0.002432
1076    0.106952
1101    0.013699
1106    0.005637
1116    0.024342
1126    0.007455
1127    0.008503
1153    0.048271
1178    0.011548
1191    0.023622
1217    0.015385
1225    0.019531
1243    0.051520
1245    0.002227
1364    0.005203
1371    0.002871
1388    0.026109
1390    0.013993
1402    0.016678
1421    0.266537
1427    0.011327
1429    0.012560
1439    0.009727
1466    0.019149
1471    0.021186
1473    0.0082

# EXPORT DATASET WITH THE ENGINEERED FEATURES

In [256]:
story_data.columns

Index(['Creator name', 'Creator_gender', 'Filename', 'Social permalink',
       'Channel', 'Followers', 'Type of content', 'Post creation date', 'mese',
       'Weekend/Settimanale', 'Post creation time', 'fascia_oraria',
       'Post caption', 'Reach', 'Likes', 'Comments', 'Total clicks',
       'Brand name', 'Industry', 'Local', 'Brand_SM', 'media_duration_sec',
       'face_frame_ratio', 'faccia', 'first_face_position_ratio',
       'motion_level', 'saturation', 'luminance', 'contrast', 'sharpness',
       'color_complexity', 'cognitive_overload', 'flashiness', 'PERC_REACHED',
       'CTR'],
      dtype='object')

In [257]:
reel_data.columns

Index(['Creator name', 'Creator_gender', 'Filename', 'Social permalink',
       'Channel', 'Followers', 'Type of content', 'Post creation date', 'mese',
       'Weekend/Settimanale', 'Post creation time', 'fascia_oraria',
       'Post caption', 'Reach', 'Likes', 'Comments', 'Total clicks',
       'Brand name', 'Industry', 'Local', 'Brand_SM', 'media_duration_sec',
       'face_frame_ratio', 'faccia', 'first_face_position_ratio',
       'motion_level', 'saturation', 'luminance', 'contrast', 'sharpness',
       'color_complexity', 'cognitive_overload', 'flashiness', 'PERC_REACHED',
       'ENGAGE_RATE', 'COMM_PER_LIKE'],
      dtype='object')

In [258]:
# export dataset with engineered features
story_data.to_csv("uniting_stories_engineered.csv", index=False)
reel_data.to_csv("uniting_reels_engineered.csv", index=False)

In [259]:
# FURTHER ANALYSIS:

# LMM
# linear mixed model per categoria di creator (creator sportivi, cibo, casual, ...)

# BRAVURA CREATOR
# per un brand è importante valutare quanto un influencer è bravo a fare il suo lavoro
# i.e. understand if the creator is good or if many interactions are due to the brand names attracting views
# so understand if the creator gets numbers regardless of the brand, because we think a brand would want good creators for sponsorship
# maybe as target: reach/brand_sm
# if the video is not sponsored put 1 at the denominator so you only consider the reach

# MARKOV CHAIN
# does the engagement rate of the previous post (of the same creator based on date) impact the engagement of the next post (next by date)??
# i.e. if post2 has ER2 > ER1 (ER of post1) then state 1
# otherwise state 0
# (or instead of checking ER2>ER1, we can check ER2>overall creator average, because maybe a post goes well above the average but not as much as the previous one)
# (i.e. if post1 is above the average then post2 is also above the average)
# HMM: to make them hidden we use as hidden state whether it's a story or reel

# SQUADRA CHE VINCE NON SI CAMBIA
# post1 is made with covariates X1 (those selected features, e.g. time slot, saturation, face ratio).
# if post1 performed well in terms of engagement:
# post2 immediately following post1 in terms of date will have covariates X2 similar to X1 (don't change a winning team)
# if post1 performed poorly in terms of engagement:
# post2 immediately following post1 in terms of date will have covariates X2 different from X1 (try a different combination because X1 didn't work)
# NB: but it's not certain that creators actually behave this way???
# NB2: how do we model this? markov chain? other???
# we can test (hypothesis test) if creators behave optimally: if post1 goes well then post2 stays the same, if post1 goes poorly then post2 is different
# test with hypothesis test if keeping X the same when a post goes well is the optimal strategy i.e. keeps engagement high???? (in my opinion we're overcomplicating)

# MISURARE VARIETA INFLUENCER
# measure how much the same influencer varies their posts (same X or not: time, face ratio, saturation, topic, type of caption, ...)
# how to do it????

# MORAN SCATTERPLOT
# we have a time series of a creator's posts with n posts, for each point h take k points before and k after (in terms of time series of the same creator)
# plot the moran scatterplot:
# - x axis: how much point h deviates in terms of engagement from the mean of the entire time series of the creator (higher or lower than the mean)
# - y axis: how much the neighbors deviate in terms of engagement from the mean of the entire time series of the creator (higher or lower than the mean)
# so we see high high, low low, etc